# Добавляем # slots:, если его нет. Работает, если есть # intent:
В строке `process_conll_file('<Ваш файл>', 'output.conll')` замените `<Ваш файл>` на название своего файла. Можно указать путь к файлу, но тогда надо написать так `process_conll_file(r'<Ваш файл>', 'output.conll')`. В конце у вас должен пояится файл `'output.conll'`, где будут `# slots:`. Этот файл можно переименовать.


In [52]:
def process_conll_file(input_filename, output_filename):
    with open(input_filename, "r") as f:
        read = f.read()
    lines = read.split('\n')
    output_lines = []
    found_intent = False  # Флаг для отслеживания строки intent

    # Проходим по каждой строке
    for i, line in enumerate(lines):
        # Добавляем текущую строку в список выходных строк
        output_lines.append(line)

        # Проверяем, начинается ли строка с "# intent:"
        if line.startswith('# intent:'):
            # Проверяем наличие строки # slots: после intent
            if (i + 1) < len(lines) and not lines[i + 1].startswith('# slots:'):
                # Добавляем строку # slots:
                output_lines.append('# slots:')

        # Если строка "# slots:" уже встретилась, сбрасываем флаг
        if line.startswith('# slots:'):
            found_intent = False

    # Объединяем список выходных строк в один текст
    output_text = '\n'.join(output_lines)

    # Записываем измененный текст в выходной файл
    with open(output_filename, "a") as f:
        f.write(output_text)

# Добавьте свой файл
process_conll_file('<Ваш файл>', 'output.conll')


# Проверяем на Табуляцию. Это надо сделать обязательно, потому что в следуюем коде я буду делить строку по табуляции и так вытаскивать слоты



In [53]:
# Открываем файл для чтения и последующей записи
with open('<Ваш файл>', 'r+', encoding='utf-8') as f:
    # Читаем содержимое файла
    content = f.read()
    # Разделяем содержимое файла на абзацы по пустой строке
    paragraphs = content.strip().split('\n\n')

    # Список для хранения обновленных абзацев
    updated_paragraphs = []

    # Проходим по каждому абзацу
    for paragraph in paragraphs:
        # Разделяем абзац на строки
        lines = paragraph.strip().split('\n')
        # Найдем индекс строки # slots:
        slots_line_index = -1
        for i, line in enumerate(lines):
            if line.startswith('# slots:'):
                slots_line_index = i
                break

        # Если нашли строку # slots:
        if slots_line_index != -1:
            # Начинаем проверку строк после строки # slots:
            for j in range(slots_line_index + 1, len(lines)):
                line = lines[j]
                # Разделяем строку по пробелам
                elements = line.split()
                # Если строка была разделена, то есть пробелы
                if len(elements) > 1:
                    # Объединяем элементы с табуляцией
                    tabbed_line = '\t'.join(elements)
                    # Обновляем строку с табуляцией
                    lines[j] = tabbed_line

        # Добавляем обновленный абзац в список
        updated_paragraphs.append('\n'.join(lines))

    # Переходим в начало файла и записываем обновленные абзацы
    f.seek(0)
    f.write('\n\n'.join(updated_paragraphs))
    f.truncate()  # Обрезаем остаток содержимого файла, если он есть


# Заполняем # slots.
Добавьте название своего файла в `with open('<Ваш файл>', 'r+') as f:`

# Это вариант для татарского


In [54]:
# Открываем файл для чтения и последующей записи
with open('<Ваш файл>', 'r+', encoding='utf-8') as f:
    read = f.read()
    paragraphs = read.strip().split('\n\n')

    updated_paragraphs = []

    for paragraph in paragraphs:
        slots = []
        lines = paragraph.strip().split('\n')

        # --- безопасно получаем текст ---
        tat_lines = [line for line in lines if line.startswith('# text:')]
        if not tat_lines:
            updated_paragraphs.append(paragraph)
            continue

        tat_text = tat_lines[0].split(':', 1)[1].strip()

        # --- позиции слов ---
        word_positions = []
        cursor = 0

        for word in tat_text.split():
            start = tat_text.find(word, cursor)
            end = start + len(word)
            word_positions.append((start, end))
            cursor = end

        # --- BIO ---
        current_start = None
        current_end = None
        current_mark = None

        for line in lines:
            if line.startswith('#'):
                continue

            parts = line.split('\t')
            if len(parts) != 4:
                continue

            index, word, intent, mark = parts
            idx = int(index) - 1

            if idx >= len(word_positions):
                continue

            start_char, end_char = word_positions[idx]

            if mark == 'O':
                if current_start is not None:
                    slots.append(f'{current_start}:{current_end}:{current_mark}')
                    current_start = None
                    current_end = None
                    current_mark = None
                continue

            entity_type = mark[2:]

            if mark.startswith('B-'):
                if current_start is not None:
                    slots.append(f'{current_start}:{current_end}:{current_mark}')

                current_start = start_char + 1
                current_end = end_char
                current_mark = entity_type

            elif mark.startswith('I-') and current_mark == entity_type:
                current_end = end_char

        # закрываем последний слот
        if current_start is not None:
            slots.append(f'{current_start}:{current_end}:{current_mark}')

        slots_str = ', '.join(slots)

        # --- обновляем ---
        updated_lines = []
        found_slots = False

        for line in lines:
            if line.startswith('# slots:'):
                updated_lines.append(f'# slots: {slots_str}')
                found_slots = True
            else:
                updated_lines.append(line)

        if not found_slots:
            for i, line in enumerate(updated_lines):
                if line.startswith('# intent:'):
                    updated_lines.insert(i + 1, f'# slots: {slots_str}')
                    break

        updated_paragraphs.append('\n'.join(updated_lines))

    # запись
    f.seek(0)
    f.write('\n\n'.join(updated_paragraphs))
    f.truncate()